# Retrieve NOAA NWM data

*Reproducible Workflows with Open Water Data: NASA, NOAA, USGS River Data Access*, Session 4, Thursday, November 12, 2026

**Learning goal:** Retrieve NOAA NWM data for a location of interest, describe what comes back using the course's shared concepts, and choose an access pattern that scales with your question.

The NOAA National Water Model (NWM) simulates and forecasts streamflow for about 2.8 million river reaches across the United States. You will find the reach for the Ohio River at Louisville, Kentucky, and retrieve its forecasts in Python: today's forecast, and archived forecasts run during the April 2025 flood.

The slides explain the ideas, and this notebook is for running the code. Run the cells in order.

## Before you start

**Setup.** Follow the setup instructions in the HydroShare README, then choose the **Python (open-water-data)** kernel.

**Downloads and time.** The optional `hydrotools` cell downloads about 270 MB of NWM files into `hydrotools_data/` in your working folder, and the kerchunk examples stream about 1 GB more from the cloud without saving it. The whole notebook took about 5–8 minutes in our tests, 1–2 fewer without the `hydrotools` cell; the slowest cell, in "Understanding what you downloaded", reads 240 files. The `hydrotools` cell needs about 1.7 GB of memory: it's optional, nothing later uses its result, and you can skip it in a smaller environment. Delete `hydrotools_data/` when you're done.

**Results can change.** The NOAA NWM API keeps only the last few days of forecasts, so it always returns the newest runs, and the "Going further" example reads yesterday's run. The April 2025 and February 2026 forecasts come from the archive and don't change.

## Choosing an access route

For reference while you work: the routes in this notebook, and what each is for.

NWM output is big: every forecast hour is a file covering all ~2.8 million reaches. How you should access it depends mostly on **how many forecast runs (reference times) you need**, and much less on how many reaches. Except for the CIROH BigQuery API (last row), none of these routes needs an API key.

| Use case | Recommended route | Key needed? | Scaling limits |
|---|---|---|---|
| Today's or the last few days' forecasts for a few reaches | NOAA NWM API | No | Keeps only ~3–5 days of runs; labelled experimental; built for a few reaches per request |
| Past forecasts, a few reaches, a few reference times | `hydrotools` ([Google Cloud archive](https://console.cloud.google.com/marketplace/product/noaa-public/national-water-model)) | No | Downloads whole CONUS (contiguous U.S.) files (~250–280 MB per short-range run); cost grows with the number of runs, not reaches |
| Past forecasts for many reaches or a region, or many reference times | Kerchunk references + `xarray`/`dask`, ideally run in the cloud | No | Reads only the `streamflow` chunk (~7× fewer bytes than whole files) with less memory, but each read is still a CONUS-wide chunk; building references is a one-time step per run |
| Long historical record (simulation, not forecasts) | NWM retrospective simulation (cloud Zarr) | No | A different product |
| Researchers working on CIROH projects who want to query selected forecast data without downloading whole files | [CIROH NWM BigQuery API](https://hub.ciroh.org/docs/products/data-management/bigquery-api/) | Yes, by request | You request only the data you need through a REST API backed by a BigQuery dataset, unlike `hydrotools`, which downloads whole files from the Google Cloud archive. Free for CIROH members and partners with active CIROH projects; request access and estimate query costs first (see the CIROH page). Not covered further in this notebook |

**Live coding 1: find the reach.** Run from here to the end of *Map: where these reaches are*, then go back to the slides.

## Credentials

No account or key is needed for anything in this notebook: the NLDI, the NOAA NWM API and the NWM archives on Google Cloud and AWS are all public.

## Define your query

Keep the place you are asking about in a variable at the top, rather than typing coordinates into the middle of a call. That way your query is recorded in one spot and is easy to change. For your own river, change `POINT`: it is a `(longitude, latitude)` pair in EPSG:4326 (ordinary longitude and latitude).

In [ ]:
# the place we're asking about: the Ohio River at Louisville, KY (longitude, latitude; EPSG:4326)
POINT = (-85.7991, 38.2803)

## Programmatic data discovery

### Finding COMIDs with the NLDI

The NLDI (Network Linked Data Index) is a USGS web service that links places and monitoring sites to NHDPlus, the national stream network the NWM runs on. We call it from Python with the [`pynhd`](https://docs.hyriver.io/readme/pynhd.html) package (part of HyRiver).

#### The COMID for a point

`feature_byloc` finds the NHDPlus reach whose catchment contains your point, using the NLDI `position` endpoint.

In [ ]:
from pynhd import NLDI

nldi = NLDI()
comid_gdf = nldi.feature_byloc(POINT)   # snap the point to an NHDPlus reach
comid = int(comid_gdf["comid"].iloc[0])

print(comid_gdf.drop(columns="geometry"))
print(f"COMID: {comid}")

You should see a one-row `GeoDataFrame` with the matching `comid` (the reach's geometry is dropped from the printout), and `COMID: 10164004`: the Ohio at Louisville.

#### The COMID of a reach with a monitoring site on it

If a river has a stream gage, you can also ask the NLDI for the reach the gage sits on. Here we use the ID of the gage at Louisville, `USGS-03294500`, *only as a way to name the location*: this notebook doesn't use the gage's measurements. NLDI lists stream gages as the `nwissite` source.

In [ ]:
# look up the reach by the ID of the monitoring site on it (used here only to locate the reach)
site = nldi.getfeature_byid("nwissite", "USGS-03294500")
print(site[["identifier", "name", "comid"]])

You should see one row: `USGS-03294500`, `OHIO RIVER AT LOUISVILLE, KY`, COMID `10164004`. Both routes give the same COMID, which is a useful check that you found the right reach.

#### Every reach upstream, up to 100 km

For regional questions you want a *list* of COMIDs. `navigate_byid` walks the network from a starting COMID. `upstreamTributaries` follows every tributary, and `distance` (in km) caps how far to go.

In [ ]:
# every NHDPlus reach upstream of the Louisville reach, tributaries included, up to 100 km away
upstream = nldi.navigate_byid(fsource="comid", fid=str(comid),
                              navigation="upstreamTributaries", source="flowlines", distance=100)
upstream_comids = upstream["nhdplus_comid"].astype(int).tolist()
print(f"{len(upstream_comids)} reaches upstream of (and including) the Louisville reach")

You should see `498 reaches upstream of (and including) the Louisville reach`. `upstream` is a `GeoDataFrame` of reach lines with an `nhdplus_comid` column and a `geometry` column. On the Ohio, going farther upstream adds reaches quickly (the whole basin has tens of thousands), so set `distance` to what your question needs.

#### Map: where these reaches are

Plotting what discovery returned is a quick check that you have the right river. The geometries come back with the reaches, so the map needs no other data. The cell also saves the map as `nwm-ohio-louisville-map.png` in your working folder.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
upstream.plot(ax=ax, color="#9fb7d9", linewidth=0.8)                   # all 498 upstream reaches
comid_gdf.plot(ax=ax, color="#eb6834", linewidth=4)                    # the Louisville reach
ax.plot(*POINT, marker="*", color="#eb6834", markersize=16, markeredgecolor="white")
ax.annotate(f"COMID {comid}\nOhio River at Louisville, KY", POINT, xytext=(15, -45),
            textcoords="offset points", fontsize=10)
ax.set_aspect(1 / 0.786)                                               # roughly equal km in x and y at 38° N
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("NHDPlus reaches up to 100 km upstream of Louisville", loc="left", fontsize=12)
fig.tight_layout()
fig.savefig("nwm-ohio-louisville-map.png", dpi=150)

You should see thin blue river lines branching to the northeast and east of Louisville, with the main stem of the Ohio River running from the upper right to a highlighted orange reach and star at Louisville in the lower left.

**Back to the slides.**

**Live coding 2: get NWM forecasts.** Run from here to the end of *Understanding what you downloaded*, then go back to the slides. If your kernel restarted, use **Run All Above** from this cell first.

## Programmatic data downloads

All three routes below return the NWM's `streamflow`, in m³/s (the API labels it `CMS`), for each COMID.

### NOAA NWM API: today's forecasts for a few reaches

NOAA's [NWM API](https://api.water.noaa.gov/nwm/v1/docs) returns forecasts for one or more reaches as JSON (a text format that Python reads as lists and dictionaries), with no key and no files to manage. It is labelled **experimental**, and it keeps only roughly the **last 3–5 days** of forecasts. Use it for the latest forecasts, not for past events.

**The latest short-range forecast for the Ohio at Louisville.** This calls the API directly with `requests`: the URL names the reach, and `params` add the filters.

In [ ]:
import requests
import pandas as pd

comid = 10164004  # from the discovery step
url = f"https://api.water.noaa.gov/nwm/v1/streamflow/forecast/nwm_feature_id/{comid}/"

# ask for the most recent short-range forecast for this reach
resp = requests.get(url, params={"configuration": "short_range", "latest": "true"}, timeout=60)
resp.raise_for_status()                 # stop with an error if the request failed
run = resp.json()[0]                    # one entry per configuration + reference time

forecast = pd.DataFrame(run["forecast"][0]["timeseries"])
forecast["valid_datetime"] = pd.to_datetime(forecast["valid_datetime"])
print(run["configuration"], run["reference_datetime"], run["units"])
print(forecast.head())

You should see one line with `short_range`, the run's reference time (UTC) and `CMS`, then the first five rows of `valid_datetime` and `value`. Your numbers will differ: this is whatever run is newest when you ask.

The response is a list with one entry per forecast run. Each entry has `nwm_feature_id` (the COMID), `configuration`, `reference_datetime`, `data_type`, `units` and a `forecast` list with one item per ensemble member (`member_id`). Each member holds a `timeseries` of `valid_datetime`/`value` pairs: 18 hourly values for a short-range run. To ask about several reaches at once, separate the COMIDs with commas in the URL.

**Every recent medium-range run, all ensemble members.** Leave out `latest` and use `min_reference_datetime` (and `max_reference_datetime`) to get several runs in one request.

In [ ]:
# every medium-range run initialized in the last 2 days (the API only keeps the last few days)
since = (pd.Timestamp.now(tz="UTC") - pd.Timedelta("2D")).strftime("%Y-%m-%dT%H:00:00Z")
resp = requests.get(url, params={"configuration": "medium_range", "min_reference_datetime": since}, timeout=120)
resp.raise_for_status()

# flatten the nested JSON into one row per run, member and valid time
rows = []
for run in resp.json():
    for member in run["forecast"]:
        for step in member["timeseries"]:
            rows.append({"reference_time": run["reference_datetime"], "member": member["member_id"],
                         "valid_time": step["valid_datetime"], "streamflow_cms": step["value"]})

runs = pd.DataFrame(rows)
print(runs.groupby("reference_time")["member"].nunique())

You should see each medium-range run from the last 2 days with `6` members. `runs` is a long-format table (one row per value) with columns `reference_time`, `member`, `valid_time` and `streamflow_cms`. Ensemble members are parallel runs from slightly different inputs. Member 1 is also the single 10-day (deterministic) run, out to 240 hours; members 2–6 run out to 204 hours.

### `hydrotools`: past forecasts from the cloud archive

For anything older than a few days, such as the April 2025 flood, you need NOAA's archive of the raw NWM output files. The operational archive needs no key or account. [Google Cloud](https://console.cloud.google.com/marketplace/product/noaa-public/national-water-model) (`gs://national-water-model`) goes back to September 2018, and [AWS](https://registry.opendata.aws/noaa-nwm-pds/) (`s3://noaa-nwm-pds`) holds a shorter record. Each forecast hour is a separate NetCDF file covering all ~2.8 million reaches.

`hydrotools` (OWPHydroTools, from NOAA's Office of Water Prediction) finds those files, downloads them, and hands you a `pandas.DataFrame` for just the COMIDs you asked for. You never open a NetCDF file yourself.

**The short-range forecast initialized at 00Z on 6 April 2025.** This cell is optional. `.get()` downloads all 18 files of the run (about 270 MB) and needs about 1.7 GB of memory; it took about 2–3 minutes on our home connection. Let it finish: interrupting it can leave a broken file behind (see below). Nothing later in the notebook uses its result, so skip it if your environment is small.

In [ ]:
# Optional cell: it needs about 1.7 GB of memory, and nothing later uses its result.
# hydrotools runs its own event loop, which clashes with Jupyter's, so two steps need their own thread.
# 1. The import: the first time you import it in a folder, it downloads a crosswalk file (RouteLink.h5).
# 2. The .get() call (below). In a plain script you could drop this first block.
import importlib
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(1) as pool:
    pool.submit(importlib.import_module, "hydrotools.nwm_client.NWMFileClient").result()

from concurrent.futures import ThreadPoolExecutor
from hydrotools.nwm_client.NWMFileClient import NWMFileClient

comid = 10164004  # from the discovery step

def get_forecast():
    # downloads go to ./hydrotools_data/NWMFileClient_NetCDF_files; results are cached in ./hydrotools_data/nwm_store.parquet
    client = NWMFileClient()   # default catalog: Google Cloud
    return client.get(
        configurations=["short_range"],
        reference_times=["2025-04-06T00:00"],   # forecast reference time (UTC)
        nwm_feature_ids=[comid],
    )

# hydrotools starts its own event loop to download files, which fails inside Jupyter's running loop.
# Running the call in its own thread avoids that, and it also works in a plain script.
with ThreadPoolExecutor(1) as pool:
    forecast_data = pool.submit(get_forecast).result()

print(forecast_data[["reference_time", "nwm_feature_id", "value_time", "value", "measurement_unit"]].head().to_string())

You should see the first five rows, starting at `2025-04-06 01:00:00` with a value of about 15,580 `m3 s-1`. The result is a long-format `pandas.DataFrame`, one row per COMID and valid time (18 rows here), with columns `reference_time`, `nwm_feature_id`, `value_time`, `value`, `measurement_unit`, `variable_name`, `configuration` and `usgs_site_code` (empty, `<NA>`, for this reach). Times are in UTC, and values are in SI units.

**If `.get()` fails:**

* **With errors about reading a NetCDF/HDF file**, an earlier run was probably interrupted, leaving a half-downloaded file in `hydrotools_data/`. `hydrotools` treats files already in that folder as downloaded, so it tries to open the broken one. Delete the `hydrotools_data/` folder and run the cell again.
* **With an error about an event loop**, the first lines of the cell handle Jupyter's event loop; if you still see the error, restart the kernel and run the notebook again from the top.
* The `FutureWarning` and `UserWarning` messages printed during `.get()` are harmless: if the table prints, it worked.

`hydrotools` downloads every file of every run you ask for, in full, then extracts your COMIDs. Results are cached in `hydrotools_data/nwm_store.parquet`, so asking for the same run again is fast. Keep `hydrotools_data/` out of version control and delete it when you're done.

### Kerchunk references: lazy, cloud-native reads with `xarray`

Inside a NetCDF file, each variable is stored in compressed blocks called *chunks*. *Zarr* is a storage format that keeps each chunk as a separate object, so cloud tools can fetch exactly the chunks they need. Kerchunk scans each NetCDF file once and writes a small JSON "reference" that records *where inside the original file* each chunk of each variable lives (byte offset and length). `xarray` can then open the references as if they were one Zarr dataset and fetch **only the chunks you touch**, straight from the cloud bucket. See the [kerchunk documentation](https://fsspec.github.io/kerchunk/).

NOAA also publishes ready-made references for recent runs (since January 2026); "Going further" at the end of this notebook has an example. Here you build your own, which works for any run in the archive and takes one pass per forecast run.

**Build references for the 6 April 2025 00Z short-range run.** CIROH's [`nwmurl`](https://github.com/CIROH-UA/nwmurl) library builds NWM file URLs from a date, reference time, forecast hours, configuration and archive, so you don't have to know the archive's folder and file-naming scheme. It only builds URLs. The cell defines two helper functions, `index_one` (scan one file) and `combine_refs` (stitch the references into one dataset), and `build_refs`, which uses them.

In [ ]:
import fsspec
import nwmurl
from concurrent.futures import ThreadPoolExecutor
from kerchunk.hdf import SingleHdf5ToZarr
from kerchunk.combine import MultiZarrToZarr

def index_one(url):
    """Scan one NetCDF file's internal layout and return its byte-range references."""
    with fsspec.open(url, "rb", block_size=2**20) as f:   # read in 1 MB blocks (see below)
        return SingleHdf5ToZarr(f, url, inline_threshold=500).translate()

def combine_refs(singles):
    """Stitch per-file references into one dataset along the time dimension.
    Runs in its own thread so it also works inside Jupyter."""
    def _combine():
        return MultiZarrToZarr(singles, remote_protocol="https", concat_dims=["time"],
                               identical_dims=["feature_id", "reference_time", "crs"]).translate()
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(_combine).result()

def build_refs(day, cycle, runinput=1, meminput=None, hours=18):
    """References for one forecast run: day "YYYYMMDD", cycle = reference (initialization) hour (UTC)."""
    urls = nwmurl.generate_urls_operational(
        start_date=f"{day}0000", end_date=f"{day}0000",  # YYYYMMDDHHMM; one day
        fcst_cycle=[cycle],                 # reference time (UTC hour)
        lead_time=list(range(1, hours + 1)),  # forecast hours 1..hours
        varinput=1,                         # 1 = channel_rt (streamflow)
        geoinput=1,                         # 1 = CONUS
        runinput=runinput,                  # 1 = short_range, 2 = medium_range
        urlbaseinput=3,                     # 3 = https://storage.googleapis.com/national-water-model/
        meminput=meminput,                  # ensemble member (None for short_range)
    )
    with ThreadPoolExecutor(8) as pool:     # one request per file: run them in parallel
        singles = list(pool.map(index_one, urls))
    return combine_refs(singles)

refs = build_refs("20250406", 0)
print(len(refs["refs"]), "reference entries")

You should see `134 reference entries`. `refs` is a Python dictionary: a map from each chunk of each variable to a URL, byte offset and length. A few choices in this code are deliberate:

* `urlbaseinput=3` asks `nwmurl` for the public HTTPS address of each file in the Google Cloud archive, so both scanning and later reads use plain HTTPS with no cloud account. `nwmurl` can also point at the AWS copy (`urlbaseinput=7`) or other configurations (for example `runinput=2` and `meminput=1` for medium-range member 1, used below).
* `inline_threshold=500` stores tiny variables directly in the JSON, so the combine step doesn't need to download anything.
* `block_size=2**20` makes each file's scan one 1 MB request, which covers the file's metadata.
* Save `refs` to a JSON file and reuse it, rather than rebuilding: `import json`, then `json.dump(refs, open("refs_20250406_00z.json", "w"))`.

To check that `nwmurl` gave you files that exist, `generate_urls_operational(..., enforce_valid_outputs=1)` checks each URL first and drops missing ones.

**Read the Louisville reach from the references.**

In [ ]:
import xarray as xr

def open_refs(refs):
    """Open kerchunk references lazily: nothing is downloaded until values are needed."""
    return xr.open_dataset("reference://", engine="zarr", chunks={}, consolidated=False, zarr_format=2,
                           storage_options={"fo": refs, "remote_protocol": "https",
                                            "remote_options": {"asynchronous": True}})

ds = open_refs(refs)
print(ds)
print(ds.attrs["NWM_version_number"], ds.attrs["model_configuration"])

# pull one reach: reads 1 chunk per forecast hour (~2 MB each), not the whole files
q = ds["streamflow"].sel(feature_id=comid).load()
print(q.to_series().head())

You should see an `xarray.Dataset` with dimensions `time` (the 18 forecast hours, the valid times), `feature_id` (2,776,734 reaches) and `reference_time` (1), and variables including `streamflow` (m³/s) and `velocity`. The attributes record the model version and configuration: `v3.0 short_range`. The "2GB" size is what the full dataset *would* be in memory; nothing is downloaded until `.load()`. The first value is about 15,580 m³/s at `2025-04-06 01:00:00`: the same as `hydrotools` to within rounding.

## Understanding what you downloaded

Whichever route you used, you now have NWM streamflow forecasts for one reach. Here is what each part means, using the course's shared concepts:

| Shared concept | What it is in your download | Where to find it |
|---|---|---|
| Location identifier | COMID / `feature_id` `10164004` (the NHDPlus reach of the Ohio at Louisville) | `nwm_feature_id` (API, `hydrotools`), `feature_id` (files) |
| Variable | `streamflow`: modeled discharge through the reach | variable name in each route |
| Variable unit | m³/s | `units` = `CMS` (API), `measurement_unit` = `m3 s-1` (`hydrotools`), `ds["streamflow"].attrs["units"]` (files) |
| Time | reference time (model initialization time) and valid time (when each value applies), both UTC | `reference_datetime`/`valid_datetime` (API), `reference_time`/`value_time` (`hydrotools`), `reference_time`/`time` (files) |
| Data quality flag(s) | None. The NWM is a model, so there is no per-value flag. Judge forecasts by lead time (valid time minus reference time, one common definition) and by comparing runs | – |
| Version / provenance | NWM v3.0, configuration `short_range` (or `medium_range` member 1), from the Google Cloud or AWS archive. Forecasts from before about 18 August 2026 are v3.0; later ones are v3.1 | `ds.attrs["NWM_version_number"]`, `ds.attrs["model_configuration"]`; the API's `configuration` field |
| Data unit | One NetCDF file per forecast hour, covering every reach; a short-range run is 18 files, a medium-range member 240 | file names in the archive (e.g. `nwm.t00z.short_range.channel_rt.f001.conus.nc`) |

**What the forecasts look like.** Each forecast run is a separate forecast, so a useful picture plots several runs together. This cell builds references for two more short-range runs and for medium-range member 1 initialized on 3 April 2025 (240 files: building and reading them took 2–5 minutes in our tests, the slowest cell in the notebook), then plots them. If it runs much longer than 5 minutes, a download has probably stalled: interrupt the kernel and run the cell again. It reuses `build_refs`, `open_refs` and `comid` from above, and saves the figure as `nwm-ohio-louisville-forecasts.png`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import StrMethodFormatter

# three short-range runs (18 h each) and one medium-range run (member 1, 240 h)
short_runs = {"2025-04-06 00Z": refs, "2025-04-06 12Z": build_refs("20250406", 12),
              "2025-04-07 00Z": build_refs("20250407", 0)}
medium = build_refs("20250403", 0, runinput=2, meminput=1, hours=240)

fig, ax = plt.subplots(figsize=(8, 4.2))
q_med = open_refs(medium)["streamflow"].sel(feature_id=comid).load().to_series()
ax.plot(q_med.index, q_med.values, color="#52514e", linewidth=2, label="Medium range, initialized 2025-04-03 00Z (member 1)")
for (label, r), color in zip(short_runs.items(), ["#2a78d6", "#eb6834", "#1baf7a"]):
    q_short = open_refs(r)["streamflow"].sel(feature_id=comid).load().to_series()
    ax.plot(q_short.index, q_short.values, color=color, linewidth=2, label=f"Short range, initialized {label}")

ax.set_ylabel("Streamflow (m³/s)")
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_xlabel("Valid time, 2025 (UTC)")
ax.set_title(f"NWM v3.0 forecasts, Ohio River at Louisville (COMID {comid})", loc="left", fontsize=12)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("nwm-ohio-louisville-forecasts.png", dpi=150)

If your plot shows the 3 April medium-range run peaking near 13,900 m³/s on 8 April, and each short-range run starting near 15,000–18,000 m³/s and falling quickly, your code is right. These are model forecasts, not measurements, and different runs can tell different stories about the same river. Record exactly which runs you used, and look at several before drawing conclusions from one.

**Back to the slides.**

## Now you try it

**Now you try it: about 12 minutes.** Work through the three steps below, then go back to the slides.

Repeat the main steps for accessing NWM data on a second river: the **Willamette River at Salem, Oregon**.

**Your task:** Find the COMID for the Willamette and retrieve short-range forecasts using the NWM API and kerchunk references.

Make your changes in the code cells below. They use new names (ending in `_salem`), so your Louisville results stay as they are and you can compare them. Fill in each `...`, using the code above. If a cell stops with an error, there's probably still a `...` to fill in: the errors mention `ellipsis` or `Ellipsis`, a missing argument, or `Cannot mix str and non-str arguments`.

1. Find the COMID of the Willamette at Salem with `nldi.feature_byloc`, using the new `POINT_salem = (-123.0429, 44.9443)`.
2. Get the latest short-range forecast for that COMID (`comid_salem`) from the NOAA NWM API.
3. The Willamette ran high in late February 2026. Build kerchunk references for the short-range forecast initialized at 12Z on 25 February 2026 with `build_refs("20260225", 12)` (about 20 seconds), then read the `streamflow` for your COMID with `open_refs`.

### Step 1: find the COMID

In [ ]:
from pynhd import NLDI

POINT_salem = (-123.0429, 44.9443)        # Willamette River at Salem, OR
comid_salem = int(...)                    # TODO: look up the COMID at POINT_salem with the NLDI, as for Louisville
print("COMID:", comid_salem)

<details>
<summary><b>Show the example answer</b></summary>

```python
from pynhd import NLDI

POINT_salem = (-123.0429, 44.9443)        # Willamette River at Salem, OR
comid_salem = int(NLDI().feature_byloc(POINT_salem)["comid"].iloc[0])
print("COMID:", comid_salem)
```

</details>

**What you should get:** COMID **23791093**.

### Step 2: today's forecast

In [ ]:
import requests

url_salem = ...                           # TODO: the NOAA NWM API forecast URL for comid_salem
resp_salem = requests.get(url_salem, params=..., timeout=60)   # TODO: ask for the latest short-range forecast
resp_salem.raise_for_status()
run_salem = resp_salem.json()[0]
print(run_salem["reference_datetime"], run_salem["units"], len(run_salem["forecast"][0]["timeseries"]), "values")

<details>
<summary><b>Show the example answer</b></summary>

```python
import requests

url_salem = f"https://api.water.noaa.gov/nwm/v1/streamflow/forecast/nwm_feature_id/{comid_salem}/"
resp_salem = requests.get(url_salem, params={"configuration": "short_range", "latest": "true"}, timeout=60)
resp_salem.raise_for_status()
run_salem = resp_salem.json()[0]
print(run_salem["reference_datetime"], run_salem["units"], len(run_salem["forecast"][0]["timeseries"]), "values")
```

</details>

**What you should get:** the newest run's reference time, `CMS`, and `18 values`.

### Step 3: a past forecast

In [ ]:
refs_salem = build_refs(...)              # TODO: the short-range run initialized at 12Z on 25 February 2026
q_salem = open_refs(refs_salem)["streamflow"].sel(feature_id=comid_salem).load().to_series()
print(len(q_salem), "hourly values")
print(q_salem.iloc[[0, 8, 17]])

<details>
<summary><b>Show the example answer</b></summary>

```python
# Build references for the 2026-02-25 12Z short-range run from the Google Cloud archive
# (uses build_refs and open_refs from the kerchunk examples above)
refs_salem = build_refs("20260225", 12)
q_salem = open_refs(refs_salem)["streamflow"].sel(feature_id=comid_salem).load().to_series()
print(len(q_salem), "hourly values")
print(q_salem.iloc[[0, 8, 17]])
```

</details>

**What you should get:** 18 hourly values from your references for 25 February, with streamflow of about 1,700–1,800 m³/s, rising through the run.

**Back to the slides.**

## Going further

### Use NOAA's published references for a recent run

*Optional.* This goes beyond the session. It runs on its own (it doesn't use earlier results). "Run All" stops at the unfinished task above, so run these cells one at a time (Shift+Enter).

For runs since January 2026 you can skip the build step. NOAA's Open Data Dissemination program publishes ready-made references in `s3://noaa-nodd-kerchunk-pds` ([registry entry](https://registry.opendata.aws/noaa-nodd-kerchunk/)) for NWM `short_range` and `medium_range_mem1`, one JSON file per forecast hour, pointing at the AWS copy of the files (`s3://noaa-nwm-pds`). This example combines them and reads them over HTTPS. First, the same `open_refs` function as in "Kerchunk references":

In [ ]:
import xarray as xr

def open_refs(refs):
    """Open kerchunk references lazily: nothing is downloaded until values are needed."""
    return xr.open_dataset("reference://", engine="zarr", chunks={}, consolidated=False, zarr_format=2,
                           storage_options={"fo": refs, "remote_protocol": "https",
                                            "remote_options": {"asynchronous": True}})

In [ ]:
import json
import fsspec
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from kerchunk.combine import MultiZarrToZarr

comid = 10164004                                # Ohio River at Louisville

s3 = fsspec.filesystem("s3", anon=True)        # public bucket: no AWS account needed

# NOAA publishes one reference file per forecast hour; pick yesterday's 12Z short-range run
day = (pd.Timestamp.now(tz="UTC") - pd.Timedelta("1D")).strftime("%Y%m%d")
paths = sorted(s3.glob(f"noaa-nodd-kerchunk-pds/nwm/short_range/"
                       f"nwm.short_range.channel_rt.conus.{day}.t12z.f*.nc.zarr"))

def load_ref(path):
    """Read one NOAA reference file and point it at the same NWM files over HTTPS."""
    text = json.dumps(json.loads(s3.cat(path)))
    return json.loads(text.replace("s3://noaa-nwm-pds/", "https://noaa-nwm-pds.s3.amazonaws.com/"))

refs_list = [load_ref(p) for p in paths]

def combine():
    return MultiZarrToZarr(refs_list, remote_protocol="https", concat_dims=["time"],
                           identical_dims=["feature_id", "reference_time", "crs"]).translate()

# Run the combine step in its own thread: kerchunk starts its own event loop,
# which fails inside Jupyter's running loop.
with ThreadPoolExecutor(1) as pool:
    recent = pool.submit(combine).result()

ds_recent = open_refs(recent)
print(len(paths), "reference files;", dict(ds_recent.sizes))
print(ds_recent["streamflow"].sel(feature_id=comid).load().to_series().head(3))

**What you should see:** 18 reference files; `time` 18 and `feature_id` 2,776,734; and the first three hourly streamflow values (m³/s) at Louisville for yesterday's 12Z run (UTC), which change from day to day. Recent runs can be a newer model version than the April 2025 archive: check `ds_recent.attrs["NWM_version_number"]` and cite the version you used.

## Further reading

- NLDI documentation: [api.water.usgs.gov/docs/nldi](https://api.water.usgs.gov/docs/nldi)
- [`pynhd` (HyRiver) documentation](https://docs.hyriver.io/readme/pynhd.html); cite HyRiver as [doi:10.21105/joss.03175](https://doi.org/10.21105/joss.03175)
- [NOAA NWM API (experimental) documentation](https://api.water.noaa.gov/nwm/v1/docs) and the [National Water Prediction Service map](https://water.noaa.gov/map)
- [National Water Prediction Service API documentation](https://api.water.noaa.gov/nwps/v1/docs/)
- [OWPHydroTools (`hydrotools`) on GitHub](https://github.com/NOAA-OWP/hydrotools), the [NWM Client documentation](https://noaa-owp.github.io/hydrotools/hydrotools.nwm_client.html) and its [package page](https://pypi.org/project/hydrotools.nwm-client/)
- NWM operational archive on [Google Cloud](https://console.cloud.google.com/marketplace/product/noaa-public/national-water-model) and [AWS](https://registry.opendata.aws/noaa-nwm-pds/)
- [NOAA Cloud Optimized Zarr Reference Files (Kerchunk)](https://registry.opendata.aws/noaa-nodd-kerchunk/) and the [kerchunk documentation](https://fsspec.github.io/kerchunk/)
- [`nwmurl` (CIROH)](https://hub.ciroh.org/docs/products/data-management/dataaccess/NWMURL%20Library), which builds NWM file URLs ([source](https://github.com/CIROH-UA/nwmurl))
- Tuhinanshu, T. (2023), [*Using Kerchunk to make NOAA's National Water Model dataset more accessible*](https://element84.com/software-engineering/using-kerchunk-to-make-noaas-national-water-model-dataset-more-accessible/), Element 84
- [NWM retrospective archive (Zarr, AWS)](https://registry.opendata.aws/nwm-archive/)

## About this notebook

Released under the [GNU General Public License v3.0](https://www.gnu.org/licenses/gpl-3.0.html) (GPL-3.0). You may use, change and share it under the same license.

The `hydrotools` example is adapted from the [OWPHydroTools NWM Client README](https://github.com/NOAA-OWP/hydrotools/tree/main/python/nwm_client) (NOAA-OWP).

**How to cite the course:** Platt, L., Wong, K., Vaarre-Lamoureux, K., & Bogan, A. *Reproducible Workflows with Open Water Data: NASA, NOAA, USGS River Data Access* (Version pilot-2026-11) [Learning material (online course)]. CUAHSI. GPL-3.0.

**How to cite the data:** the operational NWM forecasts don't have a DOI. Cite the model and version (for example, NWM v3.0 for forecasts before about 18 August 2026, v3.1 after), the configuration and reference times, the archive you read from (with its URL), and the date you accessed it. Old forecasts don't change once produced, but the archives' retention can, so the access date matters.

**Acknowledgements:** this course was developed by CUAHSI with contributions from colleagues at NOAA, USGS and NASA. This material is based upon work supported by the U.S. National Science Foundation under [Award No. 2535162](https://www.nsf.gov/awardsearch/show-award?AWD_ID=2535162). Any opinions, findings, and conclusions or recommendations expressed in this material are those of the authors and do not necessarily reflect the views of the National Science Foundation. Generative AI (Anthropic's Claude) was used to help draft content, and contributors and staff reviewed, edited and tested all of it.